# AIC 2026 — Offline ASR / Audio Index v2
## Drive ZIP → `/content` → extract local → ASR local → checkpoint Drive

Notebook này dùng đúng chiến lược I/O:

```text
/content/drive/MyDrive/Videos_Lxx_*.zip
            │
            │ copy từng ZIP
            ▼
/content/aic_asr_stage/zips/
            │
            │ extract local
            ▼
/content/aic_asr_stage/videos/<zip_stem>/
            │
            │ Faster-Whisper ASR hoàn toàn trên local disk
            ▼
/content/drive/MyDrive/data/retrieval-artifacts/asr-v2/
            ├── segments/<video_id>.parquet
            ├── asr_segments.parquet
            ├── asr_video_summary.parquet
            ├── keyframe_asr_summary.parquet
            ├── progress.json
            └── manifest.json
```

Mỗi ZIP được xử lý xong rồi xóa khỏi `/content` trước khi sang ZIP kế tiếp để tránh đầy local disk.


In [1]:
# ============================================================
# 0. INSTALL
# ============================================================
!pip -q install -U "faster-whisper>=1.1.0" pyarrow unidecode tqdm
!ffmpeg -version | head -n 2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 109.4 MB/s eta 0:00:00
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)


In [2]:
# ============================================================
# 1. MOUNT + IMPORTS
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import os, gc, re, json, time, shutil, hashlib, subprocess, unicodedata, warnings, zipfile

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from unidecode import unidecode
from IPython.display import display
from faster_whisper import WhisperModel

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, "| VRAM:", round(p.total_memory / 1024**3, 1), "GB")


Mounted at /content/drive
CUDA: True
GPU: NVIDIA A100-SXM4-40GB | VRAM: 39.5 GB


In [3]:
# ============================================================
# 2. CONFIG
# ============================================================
ZIP_SOURCE_DIR = Path('/content/drive/MyDrive/data')

ZIP_NAMES = [
    'Videos_L26_c.zip',
    'Videos_L26_d.zip',
    'Videos_L26_e.zip',
    'Videos_L27_a.zip',
    'Videos_L28_a.zip',
    'Videos_L29_a.zip',
    'Videos_L30_a.zip',
]

ROOT = Path('/content/drive/MyDrive/data')
ARTIFACT_DIR = ROOT / 'retrieval-artifacts'
MASTER_PATH = ARTIFACT_DIR / 'master_keyframes.parquet'

ASR_ROOT = ARTIFACT_DIR / 'asr-v2'
SHARD_DIR = ASR_ROOT / 'segments'
PROGRESS_PATH = ASR_ROOT / 'progress.json'
SEGMENTS_PATH = ASR_ROOT / 'asr_segments.parquet'
VIDEO_SUMMARY_PATH = ASR_ROOT / 'asr_video_summary.parquet'
KEYFRAME_ASR_PATH = ASR_ROOT / 'keyframe_asr_summary.parquet'
MANIFEST_PATH = ASR_ROOT / 'manifest.json'

LOCAL_ROOT = Path('/content/aic_asr_stage')
LOCAL_ZIP_DIR = LOCAL_ROOT / 'zips'
LOCAL_VIDEO_DIR = LOCAL_ROOT / 'videos'
LOCAL_AUDIO_DIR = LOCAL_ROOT / 'audio'

MODEL_CACHE_DIR = ROOT / '.hf-cache' / 'faster-whisper'

for p in [ASR_ROOT, SHARD_DIR, LOCAL_ZIP_DIR, LOCAL_VIDEO_DIR, LOCAL_AUDIO_DIR, MODEL_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ASR_MODEL_SIZE = 'large-v3'
ASR_LANGUAGE = None
ASR_BEAM_SIZE = 5
ASR_VAD_FILTER = True
ASR_VAD_PARAMETERS = {'min_silence_duration_ms': 700}
ASR_CONTEXT_RADIUS_SECONDS = 10.0
MIN_SEGMENT_AVG_LOGPROB = -1.25
MAX_SEGMENT_NO_SPEECH_PROB = 0.85
AUDIO_SAMPLE_RATE = 16000

VIDEO_EXTENSIONS = {'.mp4', '.mkv', '.mov', '.avi', '.webm', '.m4v', '.mpeg', '.mpg'}
CLEAN_LOCAL_BATCH_AFTER_ASR = True

assert MASTER_PATH.exists(), MASTER_PATH

missing = [z for z in ZIP_NAMES if not (ZIP_SOURCE_DIR / z).exists()]
print("Missing ZIPs:", missing if missing else "None")


Missing ZIPs: None


In [4]:
# ============================================================
# 3. HELPERS
# ============================================================
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def norm_text(x):
    if x is None:
        return ''
    x = unicodedata.normalize('NFC', str(x)).replace('\x00', ' ')
    return re.sub(r'\s+', ' ', x).strip()

def lexical_key(x):
    x = unidecode(norm_text(x).lower())
    x = re.sub(r'[^a-z0-9]+', ' ', x)
    return re.sub(r'\s+', ' ', x).strip()

def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(tmp, path)

def file_signature(path):
    p = Path(path)
    st = p.stat()
    s = f'{p.name}|{st.st_size}|{st.st_mtime_ns}'
    return hashlib.sha256(s.encode()).hexdigest()

def human_bytes(n):
    n = float(n)
    for u in ['B','KB','MB','GB','TB']:
        if n < 1024 or u == 'TB':
            return f'{n:.2f} {u}'
        n /= 1024

def free_content_bytes():
    return shutil.disk_usage('/content').free


In [5]:
# ============================================================
# 4. MASTER KEYFRAMES
# ============================================================
master = pd.read_parquet(MASTER_PATH).copy()

need = {'global_id','video_id','keyframe_id','pts_time','frame_idx'}
missing_cols = need - set(master.columns)
if missing_cols:
    raise ValueError(f'Missing master columns: {missing_cols}')

master['global_id'] = pd.to_numeric(master['global_id'], errors='raise').astype(np.int64)
master['frame_idx'] = pd.to_numeric(master['frame_idx'], errors='raise').astype(np.int64)
master['pts_time'] = pd.to_numeric(master['pts_time'], errors='raise').astype(float)
master['video_id'] = master['video_id'].astype(str)
master['keyframe_id'] = master['keyframe_id'].astype(str)
master = master.sort_values('global_id').reset_index(drop=True)

assert master['global_id'].is_unique
MASTER_VIDEO_IDS = set(master['video_id'])

print("master rows:", len(master))
print("master videos:", master['video_id'].nunique())
display(master.head())


master rows: 177321
master videos: 873


,global_id,video_id,keyframe_id,pts_time,image_path,objects_json,object_labels,object_description,object_search_text,num_object_classes,...,has_semantic_text,metadata_coverage_score,n,fps,frame_idx,map_mode,prev_frame_text,next_frame_text,prev_pts_time,next_pts_time
0,0,L21_V001,001,0.0000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""lantern"": {""count"": 2, ""max_conf"": 0.7967387...",lantern skyscraper poster tower,lantern x2; skyscraper x7; poster x1; tower x4,lantern lantern skyscraper skyscraper skyscrap...,4,...,False,1,1,30.0,0,direct_n,,,NaN,3.0000
1,1,L21_V001,002,3.0000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""boat"": {""count"": 2, ""max_conf"": 0.6481285, ""...",boat clothing person skyscraper woman human fa...,boat x2; clothing x4; person x3; skyscraper x1...,boat boat clothing clothing clothing clothing ...,9,...,False,1,2,30.0,90,direct_n,,,0.0000,8.7000
2,2,L21_V001,003,8.7000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""clothing"": {""count"": 3, ""max_conf"": 0.659825...",clothing person man woman boat human face skys...,clothing x3; person x2; man x2; woman x1; boat...,clothing clothing clothing person person man m...,9,...,False,1,3,30.0,261,direct_n,,,3.0000,11.7333
3,3,L21_V001,004,11.7333,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""pumpkin"": {""count"": 1, ""max_conf"": 0.7350670...",pumpkin clothing lantern,pumpkin x1; clothing x1; lantern x1,pumpkin clothing lantern,3,...,False,1,4,30.0,351,direct_n,,,8.7000,13.7000
4,4,L21_V001,005,13.7000,/content/drive/MyDrive/data/keyframes/L21_V001...,"{""house"": {""count"": 7, ""max_conf"": 0.70072985,...",house tree plant porch,house x7; tree x9; plant x4; porch x1,house house house house house tree tree tree t...,4,...,False,1,5,30.0,411,direct_n,,,11.7333,17.7000


## 5. Inventory chỉ đọc metadata của ZIP

Cell này chỉ đọc central directory của ZIP trên Drive để biết `video_id → archive/member`.
Nó **không dùng video trực tiếp từ Drive để ASR**.


In [14]:
ZIP_NAMES = [
    'Videos_L26_b.zip',
]

In [15]:
# ============================================================
# 5. ZIP INVENTORY
# ============================================================
def zip_video_inventory(zip_name):
    zp = ZIP_SOURCE_DIR / zip_name
    sig = file_signature(zp)
    rows = []
    with zipfile.ZipFile(zp, 'r') as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            if Path(info.filename).suffix.lower() not in VIDEO_EXTENSIONS:
                continue
            rows.append({
                'archive': zip_name,
                'archive_signature': sig,
                'member_path': info.filename,
                'video_id': Path(info.filename).stem,
                'compressed_size': int(info.compress_size),
                'uncompressed_size': int(info.file_size),
                'crc': int(info.CRC),
            })
    return rows

inventory_rows = []
for z in tqdm(ZIP_NAMES, desc='ZIP inventory'):
    zp = ZIP_SOURCE_DIR / z
    if zp.exists():
        inventory_rows.extend(zip_video_inventory(z))

inventory = pd.DataFrame(inventory_rows)
if inventory.empty:
    raise RuntimeError('No videos found in configured ZIP files.')

inventory['in_master'] = inventory['video_id'].isin(MASTER_VIDEO_IDS)
ARCHIVE_VIDEO_IDS = set(inventory['video_id'])

dups = inventory.groupby('video_id').size()
dups = dups[dups > 1]

print("archive video files:", len(inventory))
print("unique video ids:", inventory['video_id'].nunique())
print("matching master:", inventory.loc[inventory.in_master, 'video_id'].nunique())
print("duplicates:", len(dups))
print("master videos absent from these ZIPs:", len(MASTER_VIDEO_IDS - ARCHIVE_VIDEO_IDS))
display(inventory.head(20))


ZIP inventory:   0%|          | 0/1 [00:00<?, ?it/s]

archive video files: 100
unique video ids: 100
matching master: 100
duplicates: 0
master videos absent from these ZIPs: 773


,archive,archive_signature,member_path,video_id,compressed_size,uncompressed_size,crc,in_master
0,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V100.mp4,L26_V100,66548101,66548101,2097636014,True
1,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V101.mp4,L26_V101,70802929,70802929,812731039,True
2,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V102.mp4,L26_V102,56877036,56877036,2300657888,True
3,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V103.mp4,L26_V103,71447376,71447376,444823312,True
4,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V104.mp4,L26_V104,75779375,75779375,2213729497,True
5,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V105.mp4,L26_V105,70566774,70566774,3768777406,True
6,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V106.mp4,L26_V106,65208825,65208825,440484326,True
7,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V107.mp4,L26_V107,70117146,70117146,2656748014,True
8,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V108.mp4,L26_V108,69179638,69179638,4026033762,True
9,Videos_L26_b.zip,9ff969cd296391e26c116da2229f869b71a3a65b79f1de...,video/L26_V109.mp4,L26_V109,79696700,79696700,3064078365,True


In [7]:
# ============================================================
# 6. LOCAL STAGING
# ============================================================
def copy_zip_to_content(zip_name):
    src = ZIP_SOURCE_DIR / zip_name
    dst = LOCAL_ZIP_DIR / zip_name

    if not src.exists():
        raise FileNotFoundError(src)

    if dst.exists():
        dst.unlink()

    required = int(src.stat().st_size * 1.10)
    if free_content_bytes() < required:
        raise RuntimeError(
            f'Not enough /content space. ZIP={human_bytes(src.stat().st_size)}, '
            f'free={human_bytes(free_content_bytes())}'
        )

    print(f'[COPY] {src} -> {dst}')
    t0 = time.time()
    shutil.copy2(src, dst)

    if dst.stat().st_size != src.stat().st_size:
        raise RuntimeError(f'Copy size mismatch: {zip_name}')

    print(f'[COPY DONE] {human_bytes(dst.stat().st_size)} in {time.time()-t0:.1f}s')
    return dst

def extract_zip_local(local_zip):
    local_zip = Path(local_zip)
    batch_dir = LOCAL_VIDEO_DIR / local_zip.stem

    if batch_dir.exists():
        shutil.rmtree(batch_dir)
    batch_dir.mkdir(parents=True, exist_ok=True)

    root = batch_dir.resolve()
    print(f'[EXTRACT] {local_zip} -> {batch_dir}')
    t0 = time.time()

    with zipfile.ZipFile(local_zip, 'r') as zf:
        for info in zf.infolist():
            target = (batch_dir / info.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member: {info.filename}')
        zf.extractall(batch_dir)

    print(f'[EXTRACT DONE] {time.time()-t0:.1f}s')
    return batch_dir

def local_videos(batch_dir):
    return sorted(
        [p for p in Path(batch_dir).rglob('*')
         if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS],
        key=lambda p: (p.stem, str(p))
    )

def cleanup_batch(zip_name):
    batch_dir = LOCAL_VIDEO_DIR / Path(zip_name).stem
    local_zip = LOCAL_ZIP_DIR / zip_name

    if batch_dir.exists():
        shutil.rmtree(batch_dir, ignore_errors=True)
    if local_zip.exists():
        local_zip.unlink()

    for wav in LOCAL_AUDIO_DIR.glob('*.wav'):
        try:
            wav.unlink()
        except OSError:
            pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [8]:
# ============================================================
# 7. AUDIO EXTRACTION
# ============================================================
def has_audio(video_path):
    cmd = [
        'ffprobe','-v','error','-select_streams','a:0',
        '-show_entries','stream=codec_type','-of','csv=p=0',
        str(video_path)
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        return False, norm_text(r.stderr)
    return bool(r.stdout.strip()), ''

def extract_audio(video_path, wav_path):
    wav_path = Path(wav_path)
    if wav_path.exists():
        wav_path.unlink()

    cmd = [
        'ffmpeg','-hide_banner','-loglevel','error','-y',
        '-i',str(video_path),'-vn','-ac','1','-ar',str(AUDIO_SAMPLE_RATE),
        '-c:a','pcm_s16le',str(wav_path)
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)

    if r.returncode != 0:
        if wav_path.exists():
            wav_path.unlink()
        raise RuntimeError(f'ffmpeg failed: {norm_text(r.stderr)[:800]}')

    if not wav_path.exists() or wav_path.stat().st_size <= 44:
        raise RuntimeError(f'Invalid WAV: {wav_path}')

    return wav_path


In [9]:
# ============================================================
# 8. LOAD ASR MODEL
# ============================================================
if torch.cuda.is_available():
    ASR_DEVICE = 'cuda'
    ASR_COMPUTE_TYPE = 'float16'
else:
    ASR_DEVICE = 'cpu'
    ASR_COMPUTE_TYPE = 'int8'
    warnings.warn('GPU unavailable; full ASR will be slow.')

asr_model = WhisperModel(
    ASR_MODEL_SIZE,
    device=ASR_DEVICE,
    compute_type=ASR_COMPUTE_TYPE,
    download_root=str(MODEL_CACHE_DIR),
    cpu_threads=max(1, min(8, os.cpu_count() or 4)),
)

print("Loaded:", ASR_MODEL_SIZE, ASR_DEVICE, ASR_COMPUTE_TYPE)


Loaded: large-v3 cuda float16


In [10]:
# ============================================================
# 9. TRANSCRIBE ONE LOCAL VIDEO
# ============================================================
SEGMENT_COLUMNS = [
    'video_id','source_archive','segment_id','start','end','duration',
    'text','text_norm','text_lexical','avg_logprob','no_speech_prob',
    'compression_ratio','temperature','language','language_probability'
]

def transcribe_local_video(video_path, source_archive):
    video_path = Path(video_path)
    video_id = video_path.stem
    wav = LOCAL_AUDIO_DIR / f'{video_id}.wav'

    ok, err = has_audio(video_path)
    if not ok:
        return {
            'status':'no_audio',
            'video_id':video_id,
            'source_archive':source_archive,
            'language':'',
            'language_probability':np.nan,
            'duration':np.nan,
            'error':err,
            'segments':pd.DataFrame(columns=SEGMENT_COLUMNS),
        }

    try:
        extract_audio(video_path, wav)

        seg_iter, info = asr_model.transcribe(
            str(wav),
            task='transcribe',
            language=ASR_LANGUAGE,
            beam_size=ASR_BEAM_SIZE,
            condition_on_previous_text=False,
            word_timestamps=False,
            vad_filter=ASR_VAD_FILTER,
            vad_parameters=ASR_VAD_PARAMETERS if ASR_VAD_FILTER else None,
        )

        rows = []
        for i, seg in enumerate(seg_iter):
            text = norm_text(getattr(seg, 'text', ''))
            if not text:
                continue

            start = float(getattr(seg, 'start', 0.0))
            end = float(getattr(seg, 'end', start))

            rows.append({
                'video_id':video_id,
                'source_archive':source_archive,
                'segment_id':int(i),
                'start':start,
                'end':end,
                'duration':max(0.0, end-start),
                'text':text,
                'text_norm':text,
                'text_lexical':lexical_key(text),
                'avg_logprob':float(getattr(seg,'avg_logprob',np.nan)),
                'no_speech_prob':float(getattr(seg,'no_speech_prob',np.nan)),
                'compression_ratio':float(getattr(seg,'compression_ratio',np.nan)),
                'temperature':float(getattr(seg,'temperature',np.nan)),
                'language':str(getattr(info,'language','') or ''),
                'language_probability':float(getattr(info,'language_probability',np.nan)),
            })

        return {
            'status':'ok',
            'video_id':video_id,
            'source_archive':source_archive,
            'language':str(getattr(info,'language','') or ''),
            'language_probability':float(getattr(info,'language_probability',np.nan)),
            'duration':float(getattr(info,'duration',np.nan)),
            'error':'',
            'segments':pd.DataFrame(rows, columns=SEGMENT_COLUMNS),
        }

    finally:
        if wav.exists():
            wav.unlink()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [11]:
# ============================================================
# 10. RESUME / CHECKPOINT
# ============================================================
def load_progress():
    p = read_json(PROGRESS_PATH, {}) or {}
    p.setdefault('schema_version', 2)
    p.setdefault('model', ASR_MODEL_SIZE)
    p.setdefault('archives', {})
    p.setdefault('videos', {})
    return p

def source_key(archive, archive_sig, member_path, crc):
    s = f'{archive}|{archive_sig}|{member_path}|{crc}'
    return hashlib.sha256(s.encode()).hexdigest()

def video_done(video_id, expected_key, progress):
    rec = progress.get('videos', {}).get(video_id)
    if not rec or rec.get('source_key') != expected_key:
        return False
    if rec.get('status') == 'no_audio':
        return True
    if rec.get('status') == 'ok':
        return (SHARD_DIR / f'{video_id}.parquet').exists()
    return False

def save_result(result, progress, key, member_path, elapsed):
    video_id = result['video_id']

    shard_path = None
    if result['status'] == 'ok':
        shard_path = SHARD_DIR / f'{video_id}.parquet'
        result['segments'].to_parquet(shard_path, index=False)

    lp = result.get('language_probability', np.nan)
    dur = result.get('duration', np.nan)

    progress['videos'][video_id] = {
        'status':result['status'],
        'source_archive':result['source_archive'],
        'member_path':member_path,
        'source_key':key,
        'segments':int(len(result['segments'])),
        'language':result.get('language',''),
        'language_probability':None if pd.isna(lp) else float(lp),
        'duration':None if pd.isna(dur) else float(dur),
        'elapsed_seconds':float(elapsed),
        'shard_path':None if shard_path is None else str(shard_path),
        'error':result.get('error',''),
        'updated_at_utc':utc_now(),
    }
    progress['updated_at_utc'] = utc_now()
    atomic_json(PROGRESS_PATH, progress)


## 11. Smoke test

Smoke test cũng đi đúng đường ZIP local, không đọc trực tiếp video trên Drive.


In [ ]:
# ============================================================
# 11. SMOKE TEST
# ============================================================
def smoke_test_one_video(video_id=None):
    candidates = inventory[inventory['in_master']].drop_duplicates('video_id')

    if video_id is None:
        target = candidates.iloc[0]
    else:
        m = candidates[candidates['video_id'] == str(video_id)]
        if m.empty:
            raise ValueError(f'video_id not found: {video_id}')
        target = m.iloc[0]

    zip_name = target['archive']
    local_zip = None

    try:
        local_zip = copy_zip_to_content(zip_name)
        batch_dir = extract_zip_local(local_zip)

        matches = [p for p in local_videos(batch_dir) if p.stem == target['video_id']]
        if len(matches) != 1:
            raise RuntimeError(f'Expected one local video, found {len(matches)}')

        result = transcribe_local_video(matches[0], zip_name)
        print({k:v for k,v in result.items() if k != 'segments'})
        display(result['segments'].head(30))
        return result

    finally:
        cleanup_batch(zip_name)

# smoke = smoke_test_one_video()


In [12]:
# ============================================================
# 12. PROCESS ONE ARCHIVE
# ============================================================
def process_archive(zip_name, force=False, stop_after_videos=None):
    src_zip = ZIP_SOURCE_DIR / zip_name
    if not src_zip.exists():
        raise FileNotFoundError(src_zip)

    archive_sig = file_signature(src_zip)
    inv = inventory[(inventory['archive'] == zip_name) & (inventory['in_master'])].copy()
    progress = load_progress()

    old = progress.get('archives', {}).get(zip_name)
    if (
        not force and old
        and old.get('status') == 'complete'
        and old.get('archive_signature') == archive_sig
    ):
        print('[SKIP COMPLETE ARCHIVE]', zip_name)
        return progress

    processed_now = 0
    failures = []

    try:
        # 1) Drive -> /content
        local_zip = copy_zip_to_content(zip_name)

        # 2) unzip directly on /content
        batch_dir = extract_zip_local(local_zip)

        # 3) ASR local videos
        local_map = {}
        for p in local_videos(batch_dir):
            local_map.setdefault(p.stem, []).append(p)

        for row in tqdm(inv.itertuples(index=False), total=len(inv), desc=zip_name):
            vid = str(row.video_id)
            key = source_key(zip_name, archive_sig, row.member_path, int(row.crc))

            if not force and video_done(vid, key, progress):
                continue

            if stop_after_videos is not None and processed_now >= int(stop_after_videos):
                break

            matches = local_map.get(vid, [])
            if len(matches) != 1:
                err = f'Expected exactly one extracted file for {vid}, found {len(matches)}'
                progress['videos'][vid] = {
                    'status':'failed',
                    'source_archive':zip_name,
                    'member_path':row.member_path,
                    'source_key':key,
                    'error':err,
                    'updated_at_utc':utc_now(),
                }
                atomic_json(PROGRESS_PATH, progress)
                failures.append((vid, err))
                continue

            t0 = time.time()
            try:
                result = transcribe_local_video(matches[0], zip_name)
                save_result(result, progress, key, row.member_path, time.time()-t0)
                processed_now += 1
            except Exception as exc:
                err = f'{type(exc).__name__}: {exc}'
                progress['videos'][vid] = {
                    'status':'failed',
                    'source_archive':zip_name,
                    'member_path':row.member_path,
                    'source_key':key,
                    'error':err[:2000],
                    'updated_at_utc':utc_now(),
                }
                atomic_json(PROGRESS_PATH, progress)
                failures.append((vid, err))

        all_done = True
        for row in inv.itertuples(index=False):
            key = source_key(zip_name, archive_sig, row.member_path, int(row.crc))
            if not video_done(str(row.video_id), key, progress):
                all_done = False
                break

        progress['archives'][zip_name] = {
            'status':'complete' if all_done else 'partial',
            'archive_signature':archive_sig,
            'video_count':int(len(inv)),
            'processed_in_last_call':int(processed_now),
            'updated_at_utc':utc_now(),
        }
        progress['updated_at_utc'] = utc_now()
        atomic_json(PROGRESS_PATH, progress)

        print(
            f'[{zip_name}] processed_now={processed_now}, '
            f'failures={len(failures)}, complete={all_done}'
        )

        if failures:
            display(pd.DataFrame(failures, columns=['video_id','error']))

        return progress

    finally:
        if CLEAN_LOCAL_BATCH_AFTER_ASR:
            cleanup_batch(zip_name)


In [16]:
# ============================================================
# 13. RUN ALL ZIPs
# ============================================================
def run_all_archives_asr(force=False, stop_after_archives=None):
    finished_this_call = 0

    for zip_name in ZIP_NAMES:
        if not (ZIP_SOURCE_DIR / zip_name).exists():
            print('[MISSING]', zip_name)
            continue

        progress = process_archive(zip_name, force=force)

        if progress.get('archives', {}).get(zip_name, {}).get('status') == 'complete':
            finished_this_call += 1

        if stop_after_archives is not None and finished_this_call >= int(stop_after_archives):
            break

    return load_progress()

# FULL:
progress = run_all_archives_asr(force=False)

# TEST 1 ARCHIVE:
# progress = run_all_archives_asr(force=False, stop_after_archives=1)


[COPY] /content/drive/MyDrive/data/Videos_L26_b.zip -> /content/aic_asr_stage/zips/Videos_L26_b.zip
[COPY DONE] 6.37 GB in 119.4s
[EXTRACT] /content/aic_asr_stage/zips/Videos_L26_b.zip -> /content/aic_asr_stage/videos/Videos_L26_b
[EXTRACT DONE] 16.1s


Videos_L26_b.zip:   0%|          | 0/100 [00:00<?, ?it/s]

[Videos_L26_b.zip] processed_now=100, failures=0, complete=True


In [ ]:
# ============================================================
# 14. PROGRESS REPORT
# ============================================================
def show_progress():
    p = load_progress()

    adf = pd.DataFrame([
        {
            'archive':z,
            'status':p.get('archives',{}).get(z,{}).get('status','not_started'),
            'video_count':p.get('archives',{}).get(z,{}).get('video_count',0),
        }
        for z in ZIP_NAMES
    ])

    vdf = pd.DataFrame([
        {
            'video_id':vid,
            'archive':rec.get('source_archive',''),
            'status':rec.get('status',''),
            'segments':rec.get('segments',0),
            'language':rec.get('language',''),
            'elapsed_seconds':rec.get('elapsed_seconds',np.nan),
            'error':rec.get('error',''),
        }
        for vid,rec in p.get('videos',{}).items()
    ])

    display(adf)
    if not vdf.empty:
        print(vdf['status'].value_counts(dropna=False))
        display(vdf.sort_values(['archive','video_id']).head(50))
    return adf, vdf

# archive_status, video_status = show_progress()


In [ ]:
# ============================================================
# 15. MERGE SHARDS
# ============================================================
def merge_asr_shards(require_complete=True):
    progress = load_progress()
    expected = sorted(MASTER_VIDEO_IDS & ARCHIVE_VIDEO_IDS)

    incomplete = []
    seg_frames = []
    video_rows = []

    for vid in tqdm(expected, desc='Merge ASR shards'):
        rec = progress.get('videos', {}).get(vid)

        if not rec or rec.get('status') not in {'ok','no_audio'}:
            incomplete.append(vid)
            continue

        if rec['status'] == 'ok':
            shard = SHARD_DIR / f'{vid}.parquet'
            if not shard.exists():
                incomplete.append(vid)
                continue

            seg = pd.read_parquet(shard)
            seg_frames.append(seg)
            full_text = norm_text(' '.join(seg['text'].astype(str).tolist()))
            count = len(seg)
        else:
            full_text = ''
            count = 0

        video_rows.append({
            'video_id':vid,
            'source_archive':rec.get('source_archive',''),
            'asr_status':rec['status'],
            'language':rec.get('language',''),
            'language_probability':rec.get('language_probability',np.nan),
            'duration':rec.get('duration',np.nan),
            'segment_count':int(count),
            'full_text':full_text,
            'full_text_lexical':lexical_key(full_text),
        })

    if incomplete:
        msg = f'Incomplete videos: {len(incomplete)}. Sample={incomplete[:20]}'
        if require_complete:
            raise RuntimeError(msg)
        warnings.warn(msg)

    if seg_frames:
        segments = pd.concat(seg_frames, ignore_index=True)
        segments = segments.sort_values(
            ['video_id','start','end','segment_id']
        ).reset_index(drop=True)
    else:
        segments = pd.DataFrame(columns=SEGMENT_COLUMNS)

    video_summary = pd.DataFrame(video_rows).sort_values('video_id').reset_index(drop=True)

    segments.to_parquet(SEGMENTS_PATH, index=False)
    video_summary.to_parquet(VIDEO_SUMMARY_PATH, index=False)

    print("segments:", len(segments), "->", SEGMENTS_PATH)
    print("videos:", len(video_summary), "->", VIDEO_SUMMARY_PATH)
    return segments, video_summary


In [ ]:
# ============================================================
# 16. KEYFRAME ASR SUMMARY: pts_time ±10s
# ============================================================
def reliable_segments(segments):
    if segments.empty:
        return segments.copy()

    avg_ok = segments['avg_logprob'].isna() | (
        segments['avg_logprob'] >= MIN_SEGMENT_AVG_LOGPROB
    )
    speech_ok = segments['no_speech_prob'].isna() | (
        segments['no_speech_prob'] <= MAX_SEGMENT_NO_SPEECH_PROB
    )
    text_ok = segments['text'].astype(str).str.strip().ne('')
    return segments[avg_ok & speech_ok & text_ok].copy()

def build_keyframe_asr_summary(segments=None, radius=ASR_CONTEXT_RADIUS_SECONDS):
    if segments is None:
        segments = pd.read_parquet(SEGMENTS_PATH)

    good = reliable_segments(segments)

    groups = {
        str(vid): g.sort_values(['start','end']).reset_index(drop=True)
        for vid,g in good.groupby('video_id', sort=False)
    }

    rows = []

    for vid, kf in tqdm(master.groupby('video_id', sort=False), desc='ASR -> keyframes'):
        vid = str(vid)
        kf = kf.sort_values('pts_time')
        seg = groups.get(vid)

        if seg is None or seg.empty:
            for r in kf.itertuples(index=False):
                t = float(r.pts_time)
                rows.append({
                    'global_id':int(r.global_id),
                    'video_id':vid,
                    'keyframe_id':str(r.keyframe_id),
                    'frame_idx':int(r.frame_idx),
                    'pts_time':t,
                    'asr_window_start':max(0.0,t-radius),
                    'asr_window_end':t+radius,
                    'asr_segment_count':0,
                    'asr_has_speech':False,
                    'asr_text':'',
                    'asr_text_lexical':'',
                    'asr_mean_logprob':np.nan,
                    'asr_max_no_speech_prob':np.nan,
                })
            continue

        starts = seg['start'].to_numpy(float)
        ends = seg['end'].to_numpy(float)
        left = 0
        right = 0
        n = len(seg)

        for r in kf.itertuples(index=False):
            t = float(r.pts_time)
            w0 = max(0.0, t-radius)
            w1 = t+radius

            while left < n and ends[left] < w0:
                left += 1
            right = max(right, left)
            while right < n and starts[right] <= w1:
                right += 1

            idx = [j for j in range(left,right) if ends[j] >= w0]

            if idx:
                sub = seg.iloc[idx]
                text = norm_text(' '.join(sub['text'].astype(str).tolist()))
                lp = pd.to_numeric(sub['avg_logprob'], errors='coerce').mean()
                nsp = pd.to_numeric(sub['no_speech_prob'], errors='coerce').max()
            else:
                text = ''
                lp = np.nan
                nsp = np.nan

            rows.append({
                'global_id':int(r.global_id),
                'video_id':vid,
                'keyframe_id':str(r.keyframe_id),
                'frame_idx':int(r.frame_idx),
                'pts_time':t,
                'asr_window_start':w0,
                'asr_window_end':w1,
                'asr_segment_count':len(idx),
                'asr_has_speech':bool(idx and text),
                'asr_text':text,
                'asr_text_lexical':lexical_key(text),
                'asr_mean_logprob':float(lp) if pd.notna(lp) else np.nan,
                'asr_max_no_speech_prob':float(nsp) if pd.notna(nsp) else np.nan,
            })

    out = pd.DataFrame(rows).sort_values('global_id').reset_index(drop=True)
    out.to_parquet(KEYFRAME_ASR_PATH, index=False)

    print("rows:", len(out))
    print("with ASR:", int(out['asr_has_speech'].sum()))
    print("saved:", KEYFRAME_ASR_PATH)
    return out


In [ ]:
# ============================================================
# 17. VALIDATE + MANIFEST
# ============================================================
def validate_artifacts(segments, video_summary, keyframe_asr):
    assert len(keyframe_asr) == len(master)
    assert keyframe_asr['global_id'].is_unique

    a = master[['global_id','video_id','keyframe_id','frame_idx']].copy()
    b = keyframe_asr[['global_id','video_id','keyframe_id','frame_idx']].copy()

    for c in ['video_id','keyframe_id']:
        a[c] = a[c].astype(str)
        b[c] = b[c].astype(str)

    assert a.equals(b)

    if not segments.empty:
        assert (segments['end'] >= segments['start']).all()

    print("Validation: OK")
    print("ASR segments:", len(segments))
    print("video summaries:", len(video_summary))
    print("keyframes with speech:", int(keyframe_asr['asr_has_speech'].sum()))
    return True

def write_manifest():
    obj = {
        'schema_version':2,
        'created_at_utc':utc_now(),
        'io_strategy':'drive_zip -> copy_to_content -> extract_local -> asr_local -> cleanup_batch',
        'zip_source_dir':str(ZIP_SOURCE_DIR),
        'zip_names':ZIP_NAMES,
        'asr_model':ASR_MODEL_SIZE,
        'context_radius_seconds':ASR_CONTEXT_RADIUS_SECONDS,
        'master_path':str(MASTER_PATH),
        'master_rows':int(len(master)),
        'artifacts':{
            'segments':str(SEGMENTS_PATH),
            'video_summary':str(VIDEO_SUMMARY_PATH),
            'keyframe_summary':str(KEYFRAME_ASR_PATH),
            'progress':str(PROGRESS_PATH),
        },
    }
    atomic_json(MANIFEST_PATH, obj)
    print("saved:", MANIFEST_PATH)
    return obj


# 18. Thứ tự chạy

### Smoke test
```python
smoke = smoke_test_one_video()
```

### Full ASR
```python
progress = run_all_archives_asr(force=False)
```

Mỗi archive chạy theo đúng thứ tự:

```text
Drive ZIP
  ↓ copy
/content/.../<archive>.zip
  ↓ extract
/content/.../<archive>/video files
  ↓ ASR
segments/<video_id>.parquet → Drive
  ↓ cleanup local ZIP + extracted batch
archive kế tiếp
```

Nếu Colab ngắt, chạy lại:

```python
progress = run_all_archives_asr(force=False)
```

Archive complete sẽ được skip. Archive partial sẽ được copy/extract lại nhưng các video đã checkpoint sẽ được skip.

### Sau khi full ASR xong
```python
asr_segments, asr_video_summary = merge_asr_shards(require_complete=True)

keyframe_asr = build_keyframe_asr_summary(asr_segments)

validate_artifacts(
    asr_segments,
    asr_video_summary,
    keyframe_asr,
)

manifest = write_manifest()
```

Artifact dùng cho query phiên bản kế tiếp:

```text
/content/drive/MyDrive/data/retrieval-artifacts/asr-v2/keyframe_asr_summary.parquet
```
